# ARC NeuroGolf static ONNX solver

Reference layout adapted from the uploaded fill/additive-marking notebook. The task-specific modelling cell uses a semantic feature-tree or a symbolic reflection builder, not raw output-template lookup.

In [1]:
!rm -rf /kaggle/working/*
%reset -f

In [2]:
COMPETITION = '/kaggle/input/competitions/neurogolf-2026'

In [3]:
import importlib.util, subprocess, sys
missing=[p for p in ['onnx','onnxruntime','onnxscript','torch','numpy'] if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
print('dependencies ok')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 9.3 MB/s eta 0:00:00
dependencies ok


In [4]:
import json, os, time, hashlib, zipfile, glob, sys,math, random, collections, csv
from pathlib import Path
import numpy as np
import torch
import onnx
import onnxruntime as ort
import torch, torch.nn as nn, torch.nn.functional as F
from collections import defaultdict
from onnx import shape_inference

In [5]:
TASK_ID = 'task181'
CH = 10
H = W = 30
TASK_PATH = Path(COMPETITION)/f'{TASK_ID}.json'
OUTDIR = Path.cwd()/'task181_30x30_fixed'
OUTDIR.mkdir(exist_ok=True)
ONNX_PATH = OUTDIR / f'{TASK_ID}.onnx'
SUBMISSION_PATH = Path.cwd() / 'submission.zip'
SUMMARY_PATH = OUTDIR / 'task181_30x30_validation_summary.json'

In [6]:
with open(TASK_PATH, 'r') as f:
    task = json.load(f)

FORBIDDEN_OPS = {'Loop','Scan','NonZero','Unique','Script','Function'}

In [7]:


def grid_to_tensor(grid):
    arr = np.asarray(grid, dtype=np.int64)
    x = np.zeros((1, CH, H, W), dtype=np.float32)
    h, w = arr.shape
    for c in range(CH):
        x[0, c, :h, :w] = (arr == c)
    return x

def padded_expected(grid):
    arr = np.asarray(grid, dtype=np.int64)
    y = np.zeros((H, W), dtype=np.int64)
    h, w = arr.shape
    y[:h, :w] = arr
    return y

def pred_grid_from_onnx(session, grid):
    x = grid_to_tensor(grid)
    name = session.get_inputs()[0].name
    y = session.run(None, {name: x})[0]
    return y.argmax(axis=1)[0].astype(np.int64)

def crop(grid30, shape):
    h, w = shape
    return grid30[:h, :w]


In [8]:
class Task181Model(nn.Module):
    def __init__(self):
        super().__init__()
        left_basis = torch.zeros(9, H, W, dtype=torch.float32)
        right_basis = torch.zeros(9, H, W, dtype=torch.float32)
        k = 0
        for r in range(3):
            for c in range(3):
                left_basis[k, r, c] = 1.0       # target columns 0..2
                right_basis[k, r, c + 6] = 1.0  # target columns 6..8
                k += 1
        self.register_buffer('left_basis', left_basis)
        self.register_buffer('right_basis', right_basis)

    def forward(self, x):
        # x: [1, 10, 30, 30], one-hot ARC canvas.
        # Extract top color-8 pattern at rows 0:3, columns 3:6.
        top = x[:, 8, 0:3, 3:6]  # [1,3,3]
        # Horizontal mirror without using dynamic shapes.
        mir = torch.cat([top[:, :, 2:3], top[:, :, 1:2], top[:, :, 0:1]], dim=2)
        flat = mir.reshape(1, 9, 1, 1)
        left_map = torch.sum(flat * self.left_basis.reshape(1, 9, H, W), dim=1)
        right_map = torch.sum(flat * self.right_basis.reshape(1, 9, H, W), dim=1)

        # Direction flag from the top cell of the lower 4-object.
        left_flag = x[:, 4, 3, 3].reshape(1, 1, 1)
        right_flag = x[:, 4, 3, 5].reshape(1, 1, 1)
        add8 = left_map * left_flag + right_map * right_flag

        # Build proper one-hot output. Added 8 cells replace background channel 0.
        out_channels = []
        for c in range(CH):
            ch = x[:, c, :, :]
            if c == 0:
                ch = ch * (1.0 - add8)
            elif c == 8:
                ch = torch.maximum(ch, add8)
            out_channels.append(ch)
        return torch.stack(out_channels, dim=1)

model = Task181Model().eval()


In [9]:
# Fast Python reference rule used only for validation, not packaged.
def solve_reference(grid):
    a = np.asarray(grid, dtype=np.int64)
    y = np.zeros((H,W), dtype=np.int64)
    h,w = a.shape
    y[:h,:w] = a
    top = (y[0:3, 3:6] == 8)
    mir = np.fliplr(top)
    if y[3,3] == 4:
        y[0:3, 0:3] = np.where(mir, 8, y[0:3, 0:3])
    if y[3,5] == 4:
        y[0:3, 6:9] = np.where(mir, 8, y[0:3, 6:9])
    return y

for split in ['train','test','arc-gen']:
    ok = 0
    total = len(task.get(split, []))
    for ex in task.get(split, []):
        pred = solve_reference(ex['input'])
        exp = padded_expected(ex['output'])
        ok += int(np.array_equal(pred, exp))
    print(split, ok, '/', total)


train 3 / 3
test 1 / 1
arc-gen 262 / 262


In [10]:
# Export static ONNX. Batch, channel, height, and width are all fixed.
dummy = torch.zeros((1, CH, H, W), dtype=torch.float32)
dummy[0,0,:,:] = 1.0

torch.onnx.export(
    model,
    dummy,
    str(ONNX_PATH),
    input_names=['input'],
    output_names=['output'],
    opset_version=17,
    do_constant_folding=True,
    dynamic_axes=None,
    dynamo=False,
)

m = onnx.load(str(ONNX_PATH))
onnx.checker.check_model(m)
# Save shape-inferred model so the validator can see static intermediate tensor shapes.
m = shape_inference.infer_shapes(m)
onnx.save(m, str(ONNX_PATH))
print('saved', ONNX_PATH, 'bytes=', ONNX_PATH.stat().st_size)


/tmp/ipykernel_16/1895016527.py:5: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


saved /kaggle/working/task181_30x30_fixed/task181.onnx bytes= 76040


In [11]:
# ONNX graph checks.
m = onnx.load(str(ONNX_PATH))
ops = sorted({node.op_type for node in m.graph.node})
forbidden_present = sorted(set(ops) & FORBIDDEN_OPS)
empty_optional_inputs = [(node.name, node.op_type, list(node.input)) for node in m.graph.node if any(i == '' for i in node.input)]

def value_shape(v):
    t = v.type.tensor_type
    if not t.HasField('shape'):
        return None
    dims = []
    for d in t.shape.dim:
        if d.HasField('dim_value'):
            dims.append(d.dim_value)
        else:
            return None
    return dims

input_shapes = {v.name: value_shape(v) for v in m.graph.input}
output_shapes = {v.name: value_shape(v) for v in m.graph.output}
value_info_shapes = {v.name: value_shape(v) for v in m.graph.value_info}
node_outputs = [o for node in m.graph.node for o in node.output]
missing_value_info = [o for o in node_outputs if o and o not in value_info_shapes and o not in output_shapes]
non_static_value_info = [k for k,v in value_info_shapes.items() if v is None]

print('ops:', ops)
print('input_shapes:', input_shapes)
print('output_shapes:', output_shapes)
print('forbidden_present:', forbidden_present)
print('empty_optional_inputs:', empty_optional_inputs)
print('missing_value_info_count:', len(missing_value_info))
print('non_static_value_info_count:', len(non_static_value_info))
assert not forbidden_present
assert not empty_optional_inputs
assert ONNX_PATH.stat().st_size < 1_440_000
assert input_shapes == {'input': [1,10,30,30]}
assert output_shapes == {'output': [1,10,30,30]}
assert not non_static_value_info


ops: ['Add', 'Concat', 'Constant', 'Gather', 'Max', 'Mul', 'ReduceSum', 'Reshape', 'Slice', 'Sub', 'Unsqueeze']
input_shapes: {'input': [1, 10, 30, 30]}
output_shapes: {'output': [1, 10, 30, 30]}
forbidden_present: []
empty_optional_inputs: []
missing_value_info_count: 0
non_static_value_info_count: 0


In [12]:
# Raw padded ONNX validation: compare the full 30x30 output against padded expected output.
sess = ort.InferenceSession(str(ONNX_PATH), providers=['CPUExecutionProvider'])

def eval_split_raw(split, indices=None):
    examples = task.get(split, [])
    if indices is None:
        indices = range(len(examples))
    ok = 0
    total = 0
    for i in indices:
        ex = examples[i]
        pred = pred_grid_from_onnx(sess, ex['input'])
        exp = padded_expected(ex['output'])
        ok += int(np.array_equal(pred, exp))
        total += 1
    return ok, total

# Harder arc-gen split: hold out complete top-pattern signatures where possible.
def signature(ex):
    a = np.asarray(ex['input'], dtype=np.int64)
    top = tuple((a[0:3, 3:6] == 8).astype(np.int64).ravel().tolist())
    direction = 'L' if a[3,3] == 4 else 'R'
    return (direction, top)

arc = task.get('arc-gen', [])
sigs = sorted(set(signature(ex) for ex in arc), key=str)
fit_sigs = set(sigs[:max(1, int(0.4 * len(sigs)))])
fit_idx = [i for i,ex in enumerate(arc) if signature(ex) in fit_sigs]
hold_idx = [i for i,ex in enumerate(arc) if signature(ex) not in fit_sigs]

summary = {
    'task_id': TASK_ID,
    'input_shape': [1,10,30,30],
    'output_shape': [1,10,30,30],
    'onnx_size_bytes': ONNX_PATH.stat().st_size,
    'ops': ops,
    'forbidden_present': forbidden_present,
    'empty_optional_inputs': empty_optional_inputs,
    'train_raw_padded': eval_split_raw('train'),
    'visible_test_raw_padded': eval_split_raw('test'),
    'arc_gen_signature_fit_40_raw_padded': eval_split_raw('arc-gen', fit_idx),
    'arc_gen_signature_holdout_60_raw_padded': eval_split_raw('arc-gen', hold_idx),
    'arc_gen_all_raw_padded': eval_split_raw('arc-gen'),
    'num_arc_gen_signatures': len(sigs),
    'num_fit_signatures': len(fit_sigs),
    'num_holdout_signatures': len(sigs)-len(fit_sigs),
}
print(json.dumps(summary, indent=2))
with open(SUMMARY_PATH, 'w') as f:
    json.dump(summary, f, indent=2)


{
  "task_id": "task181",
  "input_shape": [
    1,
    10,
    30,
    30
  ],
  "output_shape": [
    1,
    10,
    30,
    30
  ],
  "onnx_size_bytes": 76040,
  "ops": [
    "Add",
    "Concat",
    "Constant",
    "Gather",
    "Max",
    "Mul",
    "ReduceSum",
    "Reshape",
    "Slice",
    "Sub",
    "Unsqueeze"
  ],
  "forbidden_present": [],
  "empty_optional_inputs": [],
  "train_raw_padded": [
    3,
    3
  ],
  "visible_test_raw_padded": [
    1,
    1
  ],
  "arc_gen_signature_fit_40_raw_padded": [
    104,
    104
  ],
  "arc_gen_signature_holdout_60_raw_padded": [
    158,
    158
  ],
  "arc_gen_all_raw_padded": [
    262,
    262
  ],
  "num_arc_gen_signatures": 262,
  "num_fit_signatures": 104,
  "num_holdout_signatures": 158
}


In [13]:
# Build Kaggle submission.zip with task181.onnx at archive root.
if SUBMISSION_PATH.exists():
    SUBMISSION_PATH.unlink()
with zipfile.ZipFile(SUBMISSION_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(ONNX_PATH, arcname=f'{TASK_ID}.onnx')
print('wrote', SUBMISSION_PATH)
print('zip contents:', zipfile.ZipFile(SUBMISSION_PATH).namelist())


wrote /kaggle/working/submission.zip
zip contents: ['task181.onnx']


In [14]:
ONNX_PATH

PosixPath('/kaggle/working/task181_30x30_fixed/task181.onnx')